# Diabetes Prediction using Machine Learning
> **End-to-End Classification Pipeline with Leakage-Free Preprocessing (PIMA Indians Diabetes Dataset)**

---

### Project Overview
Diabetes mellitus is a chronic metabolic disease characterized by elevated blood glucose levels. Early diagnosis is crucial for preventing long-term cardiovascular and neurological complications. 

This notebook builds a production-grade machine learning classification pipeline to predict diabetes onset based on diagnostic and physiological measurements from the **PIMA Indians Diabetes Dataset**.

### Key Workflow Highlights:
- **Domain-Specific Missing Value Handling**: Physiological parameters such as Glucose, Blood Pressure, Skin Thickness, Insulin, and BMI cannot realistically be zero in living individuals. Zero values in these columns represent missing measurements and are properly encoded as `NaN`.
- **Integrated Preprocessing Pipeline (`SimpleImputer` + `StandardScaler` + `Linear SVC`)**: Combining median imputation, standard scaling, and support vector classification within a single Scikit-Learn `Pipeline` ensures all transformations are computed strictly from training partitions, eliminating data leakage.
- **Robust Evaluation**: Evaluated using held-out test metrics (Accuracy, Precision, Recall, F1, ROC-AUC, Confusion Matrix) and verified through **5-Fold Stratified Cross-Validation**.

---

## 1. Imports and Setup
Import essential libraries for numerical analysis, data handling, machine learning pipeline construction, cross-validation, and model evaluation.

In [1]:
import numpy as np
import pandas as pd
import joblib

# Preprocessing & Pipeline
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

# Model Selection & Metrics
from sklearn import svm
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

# Display configuration
pd.set_option('display.max_columns', None)


## 2. Data Loading and Exploratory Analysis
Load the diabetes dataset and inspect its structure, dimensions, column types, and initial sample records.

In [2]:
# Load the dataset
diabetes_dataset = pd.read_csv('../Dataset/diabetes.csv')

print(f"Dataset Dimensions: {diabetes_dataset.shape[0]} rows x {diabetes_dataset.shape[1]} columns")
print(f"Target Distribution:\n{diabetes_dataset['Outcome'].value_counts().rename({0: 'Non-Diabetic (0)', 1: 'Diabetic (1)'})}")

diabetes_dataset.head()


Dataset Dimensions: 768 rows x 9 columns
Target Distribution:
Outcome
Non-Diabetic (0)    500
Diabetic (1)        268
Name: count, dtype: int64


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [3]:
# Dataset schema and column information
diabetes_dataset.info()


<class 'pandas.DataFrame'>
RangeIndex: 768 entries, 0 to 767
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Pregnancies               768 non-null    int64  
 1   Glucose                   768 non-null    int64  
 2   BloodPressure             768 non-null    int64  
 3   SkinThickness             768 non-null    int64  
 4   Insulin                   768 non-null    int64  
 5   BMI                       768 non-null    float64
 6   DiabetesPedigreeFunction  768 non-null    float64
 7   Age                       768 non-null    int64  
 8   Outcome                   768 non-null    int64  
dtypes: float64(2), int64(7)
memory usage: 54.1 KB


## 3. Data Quality and Missing/Invalid Value Handling

### Domain-Specific Value Correction
In the PIMA dataset, missing measurements were historically recorded as `0`. While zero pregnancies is biologically valid, zero values in physiological indicators are biologically impossible:
- **Glucose**: Fasting blood sugar cannot be 0 mg/dL.
- **BloodPressure**: Diastolic blood pressure cannot be 0 mm Hg.
- **SkinThickness**: Triceps skin fold thickness cannot be 0 mm.
- **Insulin**: 2-hour serum insulin cannot be 0 μU/mL.
- **BMI**: Body mass index cannot be 0 kg/m².

We replace these invalid zeros with `NaN` so the imputer can handle them properly during pipeline execution.

In [4]:
# Convert invalid zero values to NaN for physiological features
cols_with_invalid_zeros = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']

for col in cols_with_invalid_zeros:
    diabetes_dataset[col] = diabetes_dataset[col].replace(0, np.nan)

# Display missing value summary
missing_summary = pd.DataFrame({
    'Missing Count (NaN)': diabetes_dataset[cols_with_invalid_zeros].isnull().sum(),
    'Missing Percentage (%)': (diabetes_dataset[cols_with_invalid_zeros].isnull().mean() * 100).round(2)
})
display(missing_summary)


,Missing Count (NaN),Missing Percentage (%)
Glucose,5,0.65
BloodPressure,35,4.56
SkinThickness,227,29.56
Insulin,374,48.70
BMI,11,1.43


## 4. Feature and Target Preparation
Separate the input feature matrix (`X`) from the diagnostic target variable (`Y`).

In [5]:
# Separate features (X) and target (Y)
X = diabetes_dataset.drop(columns='Outcome')
Y = diabetes_dataset['Outcome']

print(f"Features shape : {X.shape}")
print(f"Target shape   : {Y.shape}")
print(f"Feature names  : {list(X.columns)}")


Features shape : (768, 8)
Target shape   : (768,)
Feature names  : ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age']


## 5. Stratified Train/Test Split
Partition the data into an 80% training set and a 20% test set using stratified sampling to preserve the disease prevalence ratio across both subsets.

In [6]:
# Stratified 80/20 train/test split
X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y,
    test_size=0.2,
    stratify=Y,
    random_state=2
)

print(f"Training set: {X_train.shape[0]} samples (Class balance: {Y_train.value_counts().to_dict()})")
print(f"Test set    : {X_test.shape[0]} samples (Class balance: {Y_test.value_counts().to_dict()})")


Training set: 614 samples (Class balance: {0: 400, 1: 214})
Test set    : 154 samples (Class balance: {0: 100, 1: 54})


## 6. Pipeline Architecture & Leakage Prevention

### Why Use a Scikit-Learn Pipeline (`SimpleImputer` + `StandardScaler` + `Linear SVC`)?

1. **`SimpleImputer(strategy='median')`**:
   - Physiological metrics like Insulin and SkinThickness exhibit right-skewed distributions. Median imputation is more robust to extreme outlier values than mean imputation.
2. **`StandardScaler()`**:
   - Support Vector Classifiers maximize the margin between classes using Euclidean distance. Differences in feature magnitudes (e.g., Insulin ~ 10-800 vs. DiabetesPedigreeFunction ~ 0.1-2.4) distort distance calculations, making scaling essential.
3. **Data Leakage Prevention**:
   - If imputation and scaling are computed on the entire dataset prior to splitting, statistics from the test set leak into the training process.
   - Encapsulating all steps in a Scikit-Learn `Pipeline` guarantees that `SimpleImputer` and `StandardScaler` compute medians and scaling parameters ($\mu, \sigma$) **strictly from `X_train`**, applying those learned parameters transformations to `X_test` and future inference data without leakage.

In [7]:
# Construct Pipeline: Median Imputation -> Standard Scaling -> Linear SVC
pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('classifier', svm.SVC(kernel='linear'))
])

print("Constructed Pipeline Structure:")
print(pipeline)


Constructed Pipeline Structure:
Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler()),
                ('classifier', SVC(kernel='linear'))])


## 7. Model Training
Fit the complete pipeline on the training dataset. The imputer and scaler learn parameters strictly from `X_train` before fitting the linear Support Vector Classifier.

In [8]:
# Fit pipeline strictly on training data
pipeline.fit(X_train, Y_train)
print("Pipeline successfully fitted on training data.")


Pipeline successfully fitted on training data.


## 8. Test Set Evaluation
Evaluate the trained pipeline on the unseen held-out test split across standard classification metrics, confusion matrix, and detailed classification report.

In [9]:
# 1. Training Set Accuracy
X_train_prediction = pipeline.predict(X_train)
training_data_accuracy = accuracy_score(Y_train, X_train_prediction)
print(f"Accuracy score on training data : {training_data_accuracy * 100:.2f}%")


Accuracy score on training data : 77.85%


In [10]:
# 2. Comprehensive Test Set Evaluation
X_test_prediction = pipeline.predict(X_test)
test_data_accuracy = accuracy_score(Y_test, X_test_prediction)

precision = precision_score(Y_test, X_test_prediction)
recall = recall_score(Y_test, X_test_prediction)
f1 = f1_score(Y_test, X_test_prediction)
y_scores = pipeline.decision_function(X_test)
roc_auc = roc_auc_score(Y_test, y_scores)

print("=" * 55)
print("           HELD-OUT TEST SET EVALUATION")
print("=" * 55)
print(f"  Accuracy            : {test_data_accuracy * 100:.2f}%")
print(f"  Precision           : {precision:.4f}")
print(f"  Recall (Sensitivity): {recall:.4f}")
print(f"  F1-Score            : {f1:.4f}")
print(f"  ROC-AUC Score       : {roc_auc:.4f}")
print("=" * 55)

# Confusion Matrix
cm = confusion_matrix(Y_test, X_test_prediction)
print("\n--- Confusion Matrix ---")
print(cm)
print(f"  True Negatives (Non-Diabetic correctly classified): {cm[0,0]}")
print(f"  False Positives (Non-Diabetic misclassified)      : {cm[0,1]}")
print(f"  False Negatives (Diabetic missed by classifier)   : {cm[1,0]}")
print(f"  True Positives (Diabetic correctly identified)    : {cm[1,1]}")

# Classification Report
print("\n--- Detailed Classification Report ---")
print(classification_report(Y_test, X_test_prediction, target_names=['Non-Diabetic (0)', 'Diabetic (1)']))


           HELD-OUT TEST SET EVALUATION
  Accuracy            : 77.27%
  Precision           : 0.7568
  Recall (Sensitivity): 0.5185
  F1-Score            : 0.6154
  ROC-AUC Score       : 0.8200

--- Confusion Matrix ---
[[91  9]
 [26 28]]
  True Negatives (Non-Diabetic correctly classified): 91
  False Positives (Non-Diabetic misclassified)      : 9
  False Negatives (Diabetic missed by classifier)   : 26
  True Positives (Diabetic correctly identified)    : 28

--- Detailed Classification Report ---
                  precision    recall  f1-score   support

Non-Diabetic (0)       0.78      0.91      0.84       100
    Diabetic (1)       0.76      0.52      0.62        54

        accuracy                           0.77       154
       macro avg       0.77      0.71      0.73       154
    weighted avg       0.77      0.77      0.76       154



## 9. 5-Fold Stratified Cross-Validation
Validate the model's stability and generalization capability across the entire dataset using **5-Fold `StratifiedKFold`**. Inside each fold, the pipeline refits both the imputer and scaler strictly on the fold's training subset.

In [11]:
# 5-Fold Stratified Cross-Validation using the Pipeline
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=2)

cv_acc = cross_val_score(pipeline, X, Y, cv=cv, scoring='accuracy')
cv_precision = cross_val_score(pipeline, X, Y, cv=cv, scoring='precision')
cv_recall = cross_val_score(pipeline, X, Y, cv=cv, scoring='recall')
cv_f1 = cross_val_score(pipeline, X, Y, cv=cv, scoring='f1')
cv_roc = cross_val_score(pipeline, X, Y, cv=cv, scoring='roc_auc')

print("=" * 55)
print("   5-FOLD STRATIFIED CROSS-VALIDATION RESULTS")
print("=" * 55)
print(f"  Accuracy : {cv_acc.mean()*100:.2f}% (+/- {cv_acc.std()*100:.2f}%)")
print(f"  Precision: {cv_precision.mean():.4f} (+/- {cv_precision.std():.4f})")
print(f"  Recall   : {cv_recall.mean():.4f} (+/- {cv_recall.std():.4f})")
print(f"  F1-Score : {cv_f1.mean():.4f} (+/- {cv_f1.std():.4f})")
print(f"  ROC-AUC  : {cv_roc.mean():.4f} (+/- {cv_roc.std():.4f})")
print("=" * 55)


   5-FOLD STRATIFIED CROSS-VALIDATION RESULTS
  Accuracy : 76.43% (+/- 2.59%)
  Precision: 0.7050 (+/- 0.0534)
  Recall   : 0.5630 (+/- 0.0702)
  F1-Score : 0.6233 (+/- 0.0499)
  ROC-AUC  : 0.8291 (+/- 0.0379)


## 10. Predictive System & Model Export

### Real-Time Inference Demo
Test the trained pipeline with a sample diagnostic vector.

In [12]:
# Sample input diagnostic data
input_data = (5, 166, 72, 19, 175, 25.8, 0.587, 51)
columns = [
    'Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness',
    'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age'
]

input_df = pd.DataFrame([input_data], columns=columns)

# Run end-to-end prediction through the Pipeline
prediction = pipeline.predict(input_df)
decision_score = pipeline.decision_function(input_df)[0]

print(f"Decision Function Score : {decision_score:.4f}")
print(f"Raw Prediction Output   : {prediction[0]}")

if prediction[0] == 0:
    print("Diagnostic Outcome      : The person is not diabetic")
else:
    print("Diagnostic Outcome      : The person is diabetic")


Decision Function Score : 0.4285
Raw Prediction Output   : 1
Diagnostic Outcome      : The person is diabetic


### Pipeline Serialization
Export the fitted pipeline containing the fitted `SimpleImputer`, `StandardScaler`, and linear `SVC` classifier for deployment in the Streamlit web application.

In [13]:
# Save the complete Pipeline (imputer + scaler + classifier)
filename = '../saved_models/diabetes_pipeline.joblib'
joblib.dump(pipeline, filename)
print(f"Pipeline successfully serialized and saved to '{filename}'")


Pipeline successfully serialized and saved to '../saved_models/diabetes_pipeline.joblib'
